# Week 3: Fashion-MNIST Clustering and PCA

This notebook compares K-means on raw pixels with K-means after PCA.

## 1. Imports and data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from torchvision import datasets
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "week1" / "week1_ml_basics.ipynb").is_file()
                     and (folder / "week5" / "week5_vae.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in the research project folder or one of its subfolders.")
WEEK3_ROOT = PROJECT_ROOT / "week3"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = WEEK3_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

seed = 42
dataset = datasets.FashionMNIST(root=str(DATA_DIR), train=True, download=True)
rng = np.random.default_rng(seed)
indices = rng.choice(len(dataset), size=10000, replace=False)
X = dataset.data.numpy()[indices].astype("float32") / 255.0
y_true = dataset.targets.numpy()[indices]
X_flat = X.reshape(len(X), -1)

print("Images:", X.shape)
print("Flattened features:", X_flat.shape)


## 2. Define the K-means evaluation function

In [ ]:
def evaluate_kmeans(features, labels, name):
    kmeans = KMeans(n_clusters=10, n_init=10, random_state=seed)
    cluster_ids = kmeans.fit_predict(features)
    return {
        "Representation": name,
        "Dimension": features.shape[1],
        "NMI": normalized_mutual_info_score(labels, cluster_ids),
        "ARI": adjusted_rand_score(labels, cluster_ids),
    }


## 3. Raw pixels and PCA representations

In [ ]:
results = []
results.append(evaluate_kmeans(X_flat, y_true, "Raw pixels"))

for dimension in [50, 20, 10]:
    X_pca = PCA(n_components=dimension, random_state=seed).fit_transform(X_flat)
    results.append(evaluate_kmeans(X_pca, y_true, f"PCA-{dimension}"))

results_df = pd.DataFrame(results)
results_df


## 4. Save the results

In [ ]:
results_df.to_csv(RESULTS_DIR / "results_pixel_pca_clustering.csv", index=False)
print("Saved results_pixel_pca_clustering.csv")


## Questions to answer

1. Classification predicts known labels, while clustering groups data without using labels.
2. Clustering does not use labels because it is an unsupervised method.
3. K is the number of clusters to find.
4. A cluster center is the mean point representing a cluster in K-means.
5. PCA projects data onto directions with large variance.
6. Raw pixels can contain redundant information and do not always represent visual similarity well.
7. Dimensionality reduction can remove redundancy and noise, making clustering easier and faster.
8. Cluster IDs are arbitrary names, so cluster 0 does not have to represent class 0.
